# Embed chunks — Qwen3-Embedding-0.6B

**Runtime → Change runtime type → T4 GPU** before running.

Steps:
1. Install deps
2. Upload `chunks.jsonl`
3. Run embedding
4. Download `embeddings.npy`

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install -q sentence-transformers

In [ ]:
# ── 2. Upload chunks.jsonl ────────────────────────────────────────────────────
from google.colab import files

print("Select your chunks.jsonl file:")
uploaded = files.upload()

# Colab saves the file to the working directory under its original name
chunks_path = list(uploaded.keys())[0]
print(f"Uploaded: {chunks_path}")

In [ ]:
# ── 3. Embed ──────────────────────────────────────────────────────────────────
import os
import json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 8
MAX_SEQ_LEN = 2048   # ~8 000 chars; truncates oversized SEBI PDF chunks
EMBED_FILE = "embeddings.npy"

# Helps with VRAM fragmentation (won't fix a genuine oversize, but reduces noise)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"GPU available: {torch.cuda.is_available()}")

texts = [json.loads(line)["text"] for line in open(chunks_path, encoding="utf-8")]
print(f"Loaded {len(texts)} chunks")

# Warn if any chunk will be truncated
long = sum(1 for t in texts if len(t) // 4 > MAX_SEQ_LEN)
if long:
    print(f"  ⚠  {long} chunk(s) exceed {MAX_SEQ_LEN} tokens and will be truncated")

model = SentenceTransformer(MODEL_ID)
model.max_seq_length = MAX_SEQ_LEN  # cap BEFORE encode — this is the key fix

# encode_document() — document side of asymmetric retrieval (no instruction prefix).
# At query time use model.encode_query() instead.
embeddings = model.encode_document(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

np.save(EMBED_FILE, embeddings.astype("float32"))
print(f"Done. Shape: {embeddings.shape}  →  {EMBED_FILE}")

In [ ]:
# ── 4. Download embeddings.npy ────────────────────────────────────────────────
from google.colab import files

files.download(EMBED_FILE)